# Downscaling validation for air and dew point temperature

In [ ]:
import datetime as dt
import os

In [ ]:
from etdataset import era5
from etdataset.icos import (
    download_icos_station,
    filter_stations_by_country_code,
    get_gpkg_file,
    load_stations_config,
)
from etdataset.validation_temp.api import (
    generate_lapse_rate_for_stations_multiprocess,
    generate_timeseries_for_stations_multiprocess,
)
from etdataset.validation_temp.metrics import (
    compute_monthly,
    compute_monthly_lr,
)
from etdataset.validation_temp.plot import (
    plot_lr_csv,
    plot_station_map,
    plot_ta_tdp_csv,
)

In [ ]:
from etdataset.logging import LoggerManager

logger = LoggerManager.get_logger(__name__)

In [ ]:
from pathlib import Path

## ICOS stations

Get a csv file with all the valid ICOS stations and their metadata

In [ ]:
# path_icos_stat = get_csv_with_valid_icos_stations() # Just one time


In [ ]:
path_icos_stat = "stations_meteoL2.csv"

Check the available stations by country number

In [ ]:
filter_stations_by_country_code(path_icos_stat, "FR")

In [ ]:
# API token from your ICOS account
auth_token = os.environ["ICOS_API_TOKEN"]

In [ ]:
download_icos_station(auth_token, ["DE-Geb", "FR-CLt"])

In [ ]:
cfg = load_stations_config(path_icos_stat)

## Retrieve the time series for ICOS, ERA5, and downscaled ERA5 datasets.

In [ ]:
output = "out"
os.makedirs(output, exist_ok=True)

In [ ]:
start_date = dt.date(2023, 11, 20)
end_date = dt.date(2023, 12, 20)

Download ERA5 datasets date by date

In [ ]:
era5.download_date_by_date(
    date1=start_date,
    date2=end_date,
    dataset=era5.ERA5Dataset.ERA5,
    output=output,
)


#### Generate time series for selected ICOS stations in parallel using multiprocessing. 


Creates a list with chosen stations

In [ ]:
stations_list = ["DE-Geb", "CH-Dav", "FR-CLt", "IT-BCi"]

Creates the stations GeoPackage from the stations list and config

In [ ]:
get_gpkg_file(stations_list, cfg, "stations_package.gpkg")

Plots an interactive map of the stations from the GeoPackage created

In [ ]:
plot_station_map("stations_package.gpkg")

The time series include ICOS measurements, ERA5 data, and downscaled ERA5 data, and are saved as CSV files.

In [ ]:
if __name__ == "__main__":
    path_icos_stat = "stations_meteoL2.csv"
    cfg = load_stations_config(path_icos_stat)
    output = "out"
    os.makedirs(output, exist_ok=True)

    res = generate_timeseries_for_stations_multiprocess(
        stations_list,
        cfg,
        output,
        start_date,
        end_date,
        "csv",
    )


Read the csv files

In [ ]:
csv_dir = Path("csv")

monthly_results = {}

for csv_file in csv_dir.glob("*_timeseries.csv"):
    station = csv_file.stem.split("_")[0].split("-")[1]
    metric = "mae"  # rmse, r2, mae, mbe, slope
    monthly_result = compute_monthly(csv_file, metric)
    monthly_results[station] = monthly_result


Plot air and dew point temperature by the selected months for each stations

In [ ]:
plot_ta_tdp_csv(csv_dir, "2023-03", "2023-04")

Metrics

In [ ]:
monthly_results

## Retrieve the time series of the observed standard atmospheric profiles and the dew point lapse rate from ICOS stations.

Download ERA5 pressure datasets date by date

In [ ]:
era5.download_date_by_date(
    date1=start_date,
    date2=end_date,
    dataset=era5.ERA5Dataset.ERA5PRESSURE,
    output=output,
)


#### Generate observed standard atmospheric and Dew Point Lapse Rates for selected ICOS stations in parallel using multiprocessing. 

In [ ]:
if __name__ == "__main__":
    path_icos_stat = "stations_meteoL2.csv"
    cfg = load_stations_config(path_icos_stat)
    output = "out"
    os.makedirs(output, exist_ok=True)

    res = generate_lapse_rate_for_stations_multiprocess(
        stations_list,
        cfg,
        output,
        start_date,
        end_date,
        "csv_lapse_rate",
    )


Read the csv files

In [ ]:
csv_dir = Path("csv_lapse_rate")

monthly_lr_results = {}

for csv_file in csv_dir.glob("*_lapse_rate.csv"):
    station = csv_file.stem.split("_")[0].split("-")[1]
    metric = "rmse"
    monthly_lr_result = compute_monthly_lr(csv_file, metric)
    monthly_lr_results[station] = monthly_lr_result

Plot the computed standart atmospehric and dew point lapse rate for each stations by the selected months

In [ ]:
plot_lr_csv(csv_dir, "2023-3", "2023-04")

Metrics

In [ ]:
monthly_lr_results